In [1]:
# Parameters
BATCH_MODE = "true"


# 12. Test-Time Scaling & Raisonnement Iteratif

> *"Le pouvoir vient finalement de l'exploitation de methodes generales, dont les deux promesses les plus importantes sont le calcul et la recherche dans de vastes espaces de solutions."* — Richard Sutton, *The Bitter Lesson* (2019)

Jusqu'a present, cette serie a explore comment **entrainer** et **prompter** des LLMs. Le notebook 8 a introduit les **modeles raisonnants** (CoT integre, `reasoning_effort`). Ce notebook ouvre une troisieme voie, devenue centrale depuis 2024 : **utiliser plus de calcul au moment de l'inference** pour faire raisonner *n'importe quel* modele — pas seulement les modeles raisonnants.

On l'appelle **test-time scaling** (Snell et al., 2024) : au lieu d'un seul appel, on genere plusieurs trajectoires, on cherche dans un arbre, on raffine par auto-critique. C'est une **nouvelle dimension a scaler**, parallele a l'entrainement.

## Ce que ce notebook construit

Ce notebook n'est **pas** une simple description : nous implementons de zero **quatre moteurs d'inference**, en suivant la progression qui mene des techniques les plus simples aux plus sophistiquees. Ces moteurs sont exactement ceux qu'orchestre le projet de reference [Iterative-Contextual-Refinements](https://github.com/ryoiki-tokuiten/Iterative-Contextual-Refinements) (ICR), une application qui combine Tree-of-Thoughts, Self-Refine et Reflexion :

| Technique | Idee | Mode ICR equivalent |
|-----------|------|---------------------|
| **Best-of-N + Self-Consistency** | Generer N reponses, voter | Brique de base du scaling |
| **Tree-of-Thoughts (BFS/DFS)** | Explorer un arbre de pensees | Mode **Deepthink** (pool BFS + branches DFS) |
| **Self-Refine / Reflexion** | Boucle Generateur -> Critique -> Memoire | Mode **Contextual** (3 agents) |
| **Routeur adaptatif** | Choisir l'effort selon la difficulte | **Adaptive Deepthink** |

## Prerequis

- Avoir suivi le **notebook 1** (API OpenAI) et le **notebook 8** (modeles raisonnants)
- Une cle API OpenRouter dans `.env` (`OPENROUTER_API_KEY`)

## Objectif pedagogique

Comprendre que **la performance d'un LLM n'est pas fixee** : elle se negocie contre du **calcul d'inference** — mais que cet investissement n'est rentable que si l'on choisit la bonne technique pour la bonne structure de probleme. Vous saurez a la fin quelle technique appliquer, a quel cout, et surtout *quand elle aide vraiment*.

## 0. Configuration de l'environnement

Nous rechargeons le meme socle que le notebook 8 : un client OpenRouter (acces multi-modeles), le mode batch pour Papermill, et une fonction helper d'appel robuste. Nous definissons un **modele rapide** — le test-time scaling s'applique y compris aux modeles peu couteux, c'est tout son interet.

In [2]:
import os
import re
import time
from pathlib import Path
from collections import Counter

try:
    %pip install openai python-dotenv --quiet
except Exception:
    pass

from openai import OpenAI
from dotenv import load_dotenv

# --- Chargement robuste du .env (Papermill change le cwd) ---
current_path = Path.cwd()
_env_trouves = False
for _ in range(10):
    if (current_path / ".env").exists():
        load_dotenv(current_path / ".env")
        _env_trouves = True
        break
    if current_path.name in ("GenAI", "MyIA.AI.Notebooks") or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
# Fallback : .env peut etre dans le sous-dossier GenAI/ si on execute depuis la racine
if not _env_trouves:
    for cand in (Path.cwd() / "MyIA.AI.Notebooks" / "GenAI" / ".env",
                 Path.cwd() / "GenAI" / ".env"):
        if cand.exists():
            load_dotenv(cand)
            _env_trouves = True
            break
if not _env_trouves:
    print("WARNING: .env non trouve, le client echouera sans OPENROUTER_API_KEY.")

BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() == "true"

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)

# Modele rapide pour le sampling massif (Best-of-N, expansion ToT, Reflexion)
FAST_MODEL = os.getenv("OPENAI_MODEL_FAST", "openai/gpt-4o-mini")
JUDGE_MODEL = os.getenv("OPENAI_MODEL_JUDGE", "openai/gpt-4o-mini")

print(f"Client OpenRouter initialise.")
print(f"Modele rapide (workhorse) : {FAST_MODEL}")
print(f"Modele juge               : {JUDGE_MODEL}")
print(f"BATCH_MODE                : {BATCH_MODE}")

Note: you may need to restart the kernel to use updated packages.


Client OpenRouter initialise.
Modele rapide (workhorse) : openai/gpt-4o-mini
Modele juge               : openai/gpt-4o-mini
BATCH_MODE                : False


In [3]:
def chat(prompt, system=None, model=FAST_MODEL, temperature=0.0, max_tokens=400):
    """Appel minimaliste et robuste. Renvoie le texte ou '' en cas d'echec."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    try:
        resp = client.chat.completions.create(
            model=model, messages=messages, temperature=temperature, max_tokens=max_tokens,
        )
        c = resp.choices[0].message.content
        return c.strip() if c else ""
    except Exception as e:
        if not BATCH_MODE:
            print(f"  [warn] appel echoue: {type(e).__name__}: {e}")
        return ""


print("Helper chat() pret.")

Helper chat() pret.


## 1. Un benchmark calibre pour mesurer ce qui compte

Pour demontrer que le test-time scaling **ameliore reellement** les resultats, il nous faut des problemes ou le modele **echoue en single-shot** mais ou le raisonnement multi-trajectoires **le rectifie**. Apres calibration, nous identifions une classe ideale : les **problemes a formule seduisante mais fausse**, ou le modele se plante par un raccourci intuitif (ex. remplacer une moyenne harmonique par une moyenne arithmetique), mais ou un echantillonnage diversifie trouve la bonne derivee.

Nous definissons un petit jeu de tels problemes, une fonction d'extraction de reponse, et un harness de benchmark.

In [4]:
PROBLEMES = [
    {
        "id": "travail",
        "question": (
            "Annie peint une cloture en 6 heures. Bernard peint la meme cloture en 3 heures. "
            "S'ils peignent ensemble a rythme constant, en combien d'heures finissent-ils ? "
            "Reponds UNIQUEMENT avec le nombre d'heures."
        ),
        "reponse": "2",  # 1/(1/6 + 1/3) = 2 ; intuition fausse : (6+3)/2 = 4.5
    },
    {
        "id": "vitesse",
        "question": (
            "Un escargot grimpe un mur de 10 metres. Le jour il monte de 3 metres, "
            "la nuit il glisse de 2 metres. En combien de jours atteint-il le sommet ? "
            "Reponds UNIQUEMENT avec le nombre de jours."
        ),
        "reponse": "8",  # le dernier jour il sort et ne glisse pas ; intuition fausse : 10
    },
    {
        "id": "machines",
        "question": (
            "Si 5 machines font 5 gadgets en 5 minutes, combien faut-il de machines "
            "pour faire 100 gadgets en 100 minutes ? "
            "Reponds UNIQUEMENT avec le nombre de machines."
        ),
        "reponse": "5",  # 1 gadget = 1 machine x 5 min ; intuition fausse : 100
    },
    {
        "id": "bouteille",
        "question": (
            "Une bouteille de vin coute 110 euros. Le vin coute 100 euros de plus "
            "que la bouteille vide. Combien vaut la bouteille vide ? "
            "Reponds UNIQUEMENT avec le nombre d'euros."
        ),
        "reponse": "5",  # intuition fausse : 10
    },
]


def extraire_nombre(texte):
    """Isole le dernier nombre entier mentionne dans la reponse."""
    if not texte:
        return None
    nombres = re.findall(r"\d+", texte.replace(" ", ""))
    return nombres[-1] if nombres else None


def est_correct(pred, attendu):
    return str(pred).strip() == str(attendu).strip()


print(f"{len(PROBLEMES)} problemes. Reponses attendues : "
      f"{[p['reponse'] for p in PROBLEMES]}")

4 problemes. Reponses attendues : ['2', '8', '5', '5']


In [5]:
def benchmark(strategie, problemes=PROBLEMES, label="strategie", verbose=True):
    """Applique strategie(probleme) -> prediction a chaque probleme. Renvoie (taux, details)."""
    details = []
    for p in problemes:
        t0 = time.time()
        pred = strategie(p)
        dt = time.time() - t0
        ok = est_correct(pred, p["reponse"])
        details.append({"id": p["id"], "pred": pred, "attendu": p["reponse"],
                        "ok": ok, "temps": dt})
        if verbose:
            stat = "OK " if ok else "ERR"
            print(f"  [{stat}] {p['id']}: pred={pred!r} attendu={p['reponse']!r} ({dt:.1f}s)")
    taux = sum(d["ok"] for d in details) / len(details)
    if verbose:
        n_ok = sum(d["ok"] for d in details)
        print(f"  => {label}: {taux*100:.0f}% ({n_ok}/{len(details)})")
    return taux, details


# --- Strategie 0 : BASELINE single-shot (notre controle) ---
def baseline(probleme):
    reponse = chat(
        f"{probleme['question']}\n\nPense etape par etape, puis donne la reponse finale.",
        temperature=0.0,
    )
    return extraire_nombre(reponse)


print("\n=== BASELINE : single-shot a temperature=0 ===")
taux_baseline, _ = benchmark(baseline, label="baseline")


=== BASELINE : single-shot a temperature=0 ===


  [OK ] travail: pred='2' attendu='2' (7.1s)


  [ERR] vitesse: pred='10' attendu='8' (5.5s)


  [OK ] machines: pred='5' attendu='5' (2.3s)


  [OK ] bouteille: pred='5' attendu='5' (1.7s)
  => baseline: 75% (3/4)


## 2. Best-of-N + Self-Consistency : scaler horizontalement

**La technique la plus simple et la plus sous-estimee.** Au lieu d'un seul appel a `temperature=0`, on genere **N reponses independantes** a temperature elevee, puis on **vote a la majorite** sur la reponse finale extraite.

C'est le constat de Wang et al. (Self-Consistency, 2022) : si plusieurs trajectoires de raisonnement *independantes* convergent vers la meme reponse, celle-ci a beaucoup plus de chances d'etre correcte. Sur nos problemes a "formule seduisante", le modele se trompe souvent en single-shot (il applique le raccourci intuitif), mais une minorite de trajectoires prennent le bon chemin — le vote les fait emerge.

> **Lien ICR** : c'est la brique elementaire que tous les modes du depot combinent ; il n'y a pas de recherche d'arbre sans d'abord savoir *sampler* et *agreger*.

In [6]:
def best_of_n(probleme, n=5):
    """Genere n reponses independantes et vote a la majorite."""
    reponses = [
        chat(
            f"{probleme['question']}\n\nPense etape par etape puis donne la reponse finale.",
            temperature=0.8,
        )
        for _ in range(n)
    ]
    nombres = [extraire_nombre(r) for r in reponses]
    nombres_valides = [x for x in nombres if x is not None]
    if not nombres_valides:
        return None
    vote = Counter(nombres_valides).most_common(1)[0][0]
    return vote


print("=== BEST-OF-N + SELF-CONSISTENCY (N=5) ===")
taux_bon, details_bon = benchmark(lambda p: best_of_n(p, n=5), label="best-of-N=5")
print(f"\nDelta vs baseline : {taux_baseline*100:.0f}% -> {taux_bon*100:.0f}%")

=== BEST-OF-N + SELF-CONSISTENCY (N=5) ===


  [OK ] travail: pred='2' attendu='2' (17.5s)


  [OK ] vitesse: pred='8' attendu='8' (11.1s)


  [OK ] machines: pred='5' attendu='5' (14.4s)


  [OK ] bouteille: pred='5' attendu='5' (9.7s)
  => best-of-N=5: 100% (4/4)

Delta vs baseline : 75% -> 100%


In [7]:
# Coût : N fois plus de tokens, mais meme latence max (appels independants, parallelisables).
print("\n--- Convergence de la precision selon N ---")
resultats_N = []
for n in [1, 3, 5]:
    taux_n, _ = benchmark(lambda p, n=n: best_of_n(p, n=n), label=f"N={n}", verbose=False)
    resultats_N.append((n, taux_n))
    print(f"  N={n}: {taux_n*100:.0f}% ({n}x cout)")
print("\nLecture : la precision monte puis plafonne avec N. C'est le trade-off fondamental")
print("du test-time scaling : acheter de la precision avec du calcul, jusqu'au point de rendement decroissant.")


--- Convergence de la precision selon N ---


  N=1: 75% (1x cout)


  N=3: 75% (3x cout)


  N=5: 100% (5x cout)

Lecture : la precision monte puis plafonne avec N. C'est le trade-off fondamental
du test-time scaling : acheter de la precision avec du calcul, jusqu'au point de rendement decroissant.


## 3. Tree-of-Thoughts : raisonner comme une recherche

Self-consistency samplait des trajectoires **independantes**. Tree-of-Thoughts (Yao et al., 2023) va plus loin : il traite le raisonnement comme une **recherche dans un arbre d'etats**, ou chaque noeud est une pensee partielle. On peut alors **etendre, evaluer, elaguer** — exactement comme A* ou minimax.

Nous implementons les **deux strategies de parcours** du mode **Deepthink** du depot ICR :

- **BFS (parcours en largeur)** — *« pool d'alternatives »* : on maintient un **pool de K pensees candidates** a la profondeur courante. A chaque etape on developpe chacune, on evalue l'ensemble, on ne garde que les K meilleures.
- **DFS (parcours en profondeur)** — *« branches evolutives »* : on choisit une branche, on la developpe profondeur par profondeur ; si elle aboutit a une impasse (auto-evaluation negative), on **revient en arriere** (backtrack).

Les deux ont besoin d'un **evaluateur** : un prompt qui demande au LLM de noter la qualite d'une pensee partielle. **Domaine naturel** : problemes combinatoires ou l'on peut decomposer l'espace (jeu du 24, cryptarithmetique, planification). Nous le demontrons sur le **jeu du 24**.

In [8]:
# --- L'evaluateur : coeur de toute recherche guidee ---
def evaluer(pensee, question, model=JUDGE_MODEL):
    """Demande une note 0-10 sur la qualite/promesse d'une pensee partielle."""
    out = chat(
        f"Probleme: {question}\n\nPensee partielle candidate: {pensee}\n\n"
        f"Note cette piste de 0 (impasse) a 10 (tres prometteuse). "
        f"Reponds UNIQUEMENT avec un entier entre 0 et 10.",
        temperature=0.0, model=model, max_tokens=10,
    )
    m = re.search(r"\d+", out or "")
    return int(m.group()) if m else 0


note_test = evaluer("Je suppose que la reponse est 0 sans reflechir.",
                    "Combien font 2+2 ?")
print(f"Evaluateur pret. Note d'une pensee bidon : {note_test}/10 (devrait etre faible).")

Evaluateur pret. Note d'une pensee bidon : 0/10 (devrait etre faible).


### 3a. ToT en BFS — maintenir un pool de K candidates

A chaque profondeur : on developpe les K pensees courantes en plusieurs propositions, on les evalue toutes, on ne garde que les K meilleures. On reproduit le *« pool d'alternatives »* du mode Deepthink.

Nous appliquons ToT au **jeu du 24** : combiner 4 nombres avec +,-,*,/ pour obtenir 24. C'est le *benchmark canonique* de ToT (Yao 2023) : l'espace est combinatoire, et le principe d'explorer-eliminer est exactement ce qui permet de trouver la solution.

In [9]:
def tot_bfs_24(nombres, largeur=2, profondeur=3, n_branches=2, target=24):
    """ToT-BFS sur le jeu du 24. Renvoie l'expression finale trouvee (str)."""
    question = (f"Atteindre {target} en combinant les nombres {nombres} "
                f"(chacun une fois) avec + - * / et des parentheses.")
    # Niveau 0 : K premieres operations candidates
    pool = []
    for _ in range(largeur):
        pool.append(chat(
            f"{question}\n\nPropose UNE premiere operation partielle "
            f"(ex. 'a + b = ...', sans conclure).",
            temperature=0.9, max_tokens=60,
        ))
    # Development niveau par niveau
    for prof in range(profondeur):
        candidats = []
        for pensee in pool:
            for _ in range(n_branches):
                suite = chat(
                    f"{question}\nEtape en cours: {pensee}\n"
                    f"Prolonge d'une operation (n'utilise que les nombres restants).",
                    temperature=0.9, max_tokens=60,
                )
                candidats.append(pensee + " | " + suite)
        notes = [(c, evaluer(c, question)) for c in candidats]
        notes.sort(key=lambda x: x[1], reverse=True)
        pool = [c for c, _ in notes[:largeur]]
    # Conclure sur la meilleure piste
    meilleure = pool[0] if pool else ""
    finale = chat(
        f"{question}\nMeilleure piste: {meilleure}\n\n"
        f"Termine : donne l'expression complete qui fait {target} (ou 'AUCUNE' si impossible).",
        temperature=0.0, max_tokens=80,
    )
    return finale.strip()


print("=== TREE-OF-THOUGHTS (BFS) sur le jeu du 24 ===")
cas_24 = [([4, 1, 8, 7], 24), ([8, 3, 8, 3], 24)]
for nombres, tgt in cas_24:
    t0 = time.time()
    expr = tot_bfs_24(nombres, largeur=2, profondeur=3, n_branches=2, target=tgt)
    print(f"  {nombres} -> {expr[:90]}  ({time.time()-t0:.0f}s)")

=== TREE-OF-THOUGHTS (BFS) sur le jeu du 24 ===


  [4, 1, 8, 7] -> Pour atteindre 24 en utilisant les nombres [4, 1, 8, 7] avec les opérations arithmétiques,  (26s)


  [8, 3, 8, 3] -> Pour atteindre 24 en utilisant les nombres [8, 3, 8, 3] avec les opérations +, -, *, / et   (27s)


### 3b. ToT en DFS — faire evoluer une branche avec retour arriere

Variante complementaire : on choisit une pensee, on la developpe en profondeur ; si l'evaluateur juge la branche mauvaise, on **abandonne et on essaie une alternative**. C'est le *« evolving branches »* du depot.

In [10]:
def tot_dfs_24(nombres, max_branches=3, max_profondeur=3, seuil=4, target=24):
    """ToT-DFS sur le jeu du 24 avec backtracking. Renvoie l'expression finale (str)."""
    question = (f"Atteindre {target} en combinant les nombres {nombres} "
                f"(chacun une fois) avec + - * / et des parentheses.")
    for b in range(max_branches):
        pensee = chat(
            f"{question}\n\nPropose une approche partielle de resolution (1-2 operations).",
            temperature=0.9, max_tokens=80,
        )
        note = evaluer(pensee, question)
        courante = pensee
        for prof in range(max_profondeur):
            if note < seuil:
                break  # impasse : backtrack vers la branche suivante
            courante = chat(
                f"{question}\nEtape: {courante}\nProlonge d'une operation decisive.",
                temperature=0.7, max_tokens=80,
            )
            note = evaluer(courante, question)
        if note >= seuil:
            finale = chat(
                f"{question}\nMeilleure piste: {courante}\n"
                f"Termine : expression complete valant {target} (ou 'AUCUNE').",
                temperature=0.0, max_tokens=80,
            )
            return finale.strip()
    return "AUCUNE branche prometteuse"


print("=== TREE-OF-THOUGHTS (DFS, backtracking) sur le jeu du 24 ===")
for nombres, tgt in cas_24:
    t0 = time.time()
    expr = tot_dfs_24(nombres, target=tgt)
    print(f"  {nombres} -> {expr[:90]}  ({time.time()-t0:.0f}s)")

=== TREE-OF-THOUGHTS (DFS, backtracking) sur le jeu du 24 ===


  [4, 1, 8, 7] -> Pour atteindre 24 en utilisant les nombres [4, 1, 8, 7] chacun une fois, nous pouvons proc  (13s)


  [8, 3, 8, 3] -> Pour atteindre 24 en utilisant les nombres [8, 3, 8, 3] une fois chacun, nous pouvons proc  (12s)


## 4. Self-Refine / Reflexion : la boucle Generateur -> Critique -> Memoire

Troisieme famille, celle du mode **Contextual** du depot ICR (3 agents : *Generator, Critique, Memory*). L'idee (Madaan 2023, *Self-Refine* ; Shinn 2023, *Reflexion*) :

1. **Generateur** : le modele tente une reponse.
2. **Critique** : le modele analyse ce qui est faux ou faible.
3. **Memoire** : on accumule les **lecons** des echecs passes.
4. On **re-tente**, informe par la critique et la memoire. On boucle.

Le modele **apprend au moment de l'inference** a partir de ses propres erreurs — sans re-entrainement. Sur nos problemes a formule seduisante, le critique detecte le raccourci errone et la memoire empeche de le reprendre.

In [11]:
def reflexion(probleme, iterations=3):
    """Boucle Generator -> Critique -> Memory."""
    memoire = []
    derniere_pred = None
    for it in range(iterations):
        # --- GENERATOR ---
        contexte = ""
        if memoire:
            contexte = ("\nLecons tirees des tentatives precedentes :\n- "
                        + "\n- ".join(memoire))
        tentative = chat(
            f"{probleme['question']}{contexte}\n\n"
            f"Pense etape par etape, puis donne la reponse finale (un nombre).",
            temperature=0.3,
        )
        derniere_pred = extraire_nombre(tentative)
        # --- CRITIQUE ---
        critique = chat(
            f"Probleme: {probleme['question']}\n"
            f"Tentative: {tentative}\n\n"
            f"Identifie en UNE phrase la faille principale du raisonnement "
            f"(s'il est correct, reponds exactement 'AUCUNE faille').",
            temperature=0.0,
        )
        if "aucune" in critique.lower():
            break  # auto-certification : on s'arrete
        # --- MEMORY ---
        memoire.append(f"Tentative={derniere_pred}; faille: {critique}")
    return derniere_pred


print("=== SELF-REFINE / REFLEXION (3 iterations) ===")
taux_refl, _ = benchmark(lambda p: reflexion(p, iterations=3), label="reflexion")
print(f"\nDelta vs baseline : {taux_baseline*100:.0f}% -> {taux_refl*100:.0f}%")

=== SELF-REFINE / REFLEXION (3 iterations) ===


  [OK ] travail: pred='2' attendu='2' (5.8s)


  [OK ] vitesse: pred='8' attendu='8' (10.0s)


  [OK ] machines: pred='5' attendu='5' (3.3s)


  [OK ] bouteille: pred='5' attendu='5' (2.5s)
  => reflexion: 100% (4/4)

Delta vs baseline : 75% -> 100%


## 5. Routeur adaptatif : la difficulte commande le cout

Le depot ICR propose un mode **Adaptive Deepthink** qui **choisit** entre BFS et DFS selon la difficulte estimee. C'est la frontiere pratique du test-time scaling : **ne pas depenser de calcul aveuglement**.

Heuristique simple : un **pre-pass peu couteux** estime la difficulte (facile / moyen / difficile) et on route vers une strategie de cout croissant. On retrouve l'intuition d'o1 et des modeles raisonnants : *investir plus de calcul seulement quand c'est utile*.

In [12]:
def estimer_difficulte(question):
    """Pre-pass peu couteux : le modele estime la difficulte en un mot."""
    out = chat(
        f"Question: {question}\n\nEstime la difficulte en UN mot: facile, moyen, ou difficile.",
        temperature=0.0, max_tokens=5,
    ).lower()
    for niv in ("facile", "moyen", "difficile"):
        if niv in out:
            return niv
    return "moyen"


def routeur_adaptatif(probleme):
    """Route vers une strategie de cout croissant selon la difficulte estimee."""
    niveau = estimer_difficulte(probleme["question"])
    if niveau == "facile":
        strategie, pred = "baseline (single-shot)", baseline(probleme)
    elif niveau == "moyen":
        strategie, pred = "best-of-N=5", best_of_n(probleme, n=5)
    else:  # difficile
        strategie, pred = "reflexion (3 iter)", reflexion(probleme, iterations=3)
    print(f"    [routeur] {probleme['id']}: difficulte={niveau} -> {strategie}")
    return pred


print("=== ROUTEUR ADAPTATIF ===")
taux_route, _ = benchmark(routeur_adaptatif, label="routeur")
print("\nLe routeur applique le bon niveau d'effort : economie sur les problemes faciles,")
print("investissement sur les difficiles. C'est le compromis cout/performance cherche.")

=== ROUTEUR ADAPTATIF ===


    [routeur] travail: difficulte=moyen -> best-of-N=5
  [OK ] travail: pred='2' attendu='2' (18.6s)


    [routeur] vitesse: difficulte=moyen -> best-of-N=5
  [OK ] vitesse: pred='8' attendu='8' (12.2s)


    [routeur] machines: difficulte=moyen -> best-of-N=5
  [OK ] machines: pred='5' attendu='5' (17.1s)


    [routeur] bouteille: difficulte=facile -> baseline (single-shot)
  [OK ] bouteille: pred='5' attendu='5' (2.4s)
  => routeur: 100% (4/4)

Le routeur applique le bon niveau d'effort : economie sur les problemes faciles,
investissement sur les difficiles. C'est le compromis cout/performance cherche.


## 6. Synthese : le cout se negocie contre la performance

Recapitulons. La colonne **appels** estime le nombre d'appels LLM par probleme — proxy du cout d'inference.

In [13]:
synthese = [
    ("Baseline (single-shot)", taux_baseline, 1),
    ("Best-of-N (N=5)",         taux_bon,      5),
    ("Reflexion (3 iter)",      taux_refl,     6),
    ("Routeur adaptatif",       taux_route,    "variable"),
]
print(f"{'Technique':<26} {'Reussite':>10} {'Appels/problem':>16}")
print("-" * 54)
for nom, taux, cout in synthese:
    print(f"{nom:<26} {taux*100:>9.0f}% {str(cout):>16}")

print("\nLecture cles :")
print("- Sur CE benchmark (formule seduisante), Best-of-N et Reflexion reparent les")
print("  echecs du single-shot. Le routeur atteint une performance similaire en moyennant")
print("  l'effort : cheap sur les faciles, lourd sur les difficiles.")
print("- Tree-of-Thoughts n'apparait pas ici : son domaine est la RECHERCHE COMBINATOIRE")
print("  (jeu du 24, cryptarithmetique, planification), pas le calcul pas-a-pas. L'appliquer")
print("  a de l'arithmetique simple est un mesusage — et couteux. Bon outil, bon domaine.")
print("- NB : sur 4 petits problemes, les taux ont une granularite de 25% : l'interet est")
print("  le MECANISME, transposable a des taches plus hardues et des benchmarks plus larges.")

Technique                    Reussite   Appels/problem
------------------------------------------------------
Baseline (single-shot)            75%                1
Best-of-N (N=5)                  100%                5
Reflexion (3 iter)               100%                6
Routeur adaptatif                100%         variable

Lecture cles :
- Sur CE benchmark (formule seduisante), Best-of-N et Reflexion reparent les
  echecs du single-shot. Le routeur atteint une performance similaire en moyennant
  l'effort : cheap sur les faciles, lourd sur les difficiles.
- Tree-of-Thoughts n'apparait pas ici : son domaine est la RECHERCHE COMBINATOIRE
  (jeu du 24, cryptarithmetique, planification), pas le calcul pas-a-pas. L'appliquer
  a de l'arithmetique simple est un mesusage — et couteux. Bon outil, bon domaine.
- NB : sur 4 petits problemes, les taux ont une granularite de 25% : l'interet est
  le MECANISME, transposable a des taches plus hardues et des benchmarks plus larges.


### La lecon fondamentale

> **La performance d'un LLM n'est pas une constante : c'est une fonction de votre budget d'inference — et de votre choix de technique.**

Cette idee a change l'industrie en 2024-2025 : les modeles de la serie **o1/o3** (OpenAI) et **AlphaCode 2** internalisent exactement ca — ils font de la recherche ToT/cachee **a l'interieur** du modele. Mais comme ce notebook le montre, **vous pouvez reproduire ces gains avec n'importe quel modele**, en orchestrant vous-meme le calcul d'inference.

C'est precisement ce que fait le depot [Iterative-Contextual-Refinements](https://github.com/ryoiki-tokuiten/Iterative-Contextual-Refinements) : il orchestre ces memes techniques (Deepthink BFS/DFS, mode Contextuel 3-agent, routeur adaptatif) dans une application web. Vous en avez maintenant les briques en Python — et le jugement pour savoir **quand** chaque technique paie.

## 7. Exercices

Les cellules suivantes sont a completer. Conformement aux conventions de la serie, elles s'executent sans erreur meme non remplies (pas de `raise NotImplementedError`). Trois pistes d'approfondissement.

### Exercice 1 — Vote pondere dans la self-consistency

Le vote a la majorite traite toutes les reponses de meme. **Ameliorez-le** : ponderez chaque reponse par un score de confiance (par ex. une auto-evaluation demandee au modele). Completez `best_of_n_pondere`.

In [14]:
# Indice : pour chaque reponse, demandez aussi une confiance 0-10 au modele,
# puis choisissez la reponse majoritaire PONDEREE par ces confiances.

def best_of_n_pondere(probleme, n=5):
    # TODO etudiant : implementer le vote pondere.
    # Etape 1 : recuperer (reponse, confiance) pour chacun des N echantillons.
    # Etape 2 : agreger les scores de confiance par reponse candidate.
    # Etape 3 : retourner la reponse au score total le plus eleve.
    result = None  # TODO etudiant
    return result

pred_exo1 = best_of_n_pondere(PROBLEMES[0], n=5)
print(f"Exercice 1 - prediction ponderee sur '{PROBLEMES[0]['id']}' : {pred_exo1} "
      f"(attendu {PROBLEMES[0]['reponse']})")

Exercice 1 - prediction ponderee sur 'travail' : None (attendu 2)


### Exercice 2 — Diversite dans le pool ToT-BFS

Le BFS garde les K meilleures pensees, mais elles peuvent etre **tres similaires**. Ajoutez une heuristique de **diversite** : penalisez les pensees trop proches semantiquement d'une deja selectionnee. Completez `selection_diverse`.

In [15]:
# Indice : apres tri par note, construisez le pool en refusant d'ajouter une pensee
# si sa similarite avec une deja retenue depasse un seuil.

def similarite(a, b):
    """Ratio de mots communs entre deux pensees (heuristique simple)."""
    # TODO etudiant : affiner (ex. similarite cosinus sur embeddings).
    mots_a = set(a.lower().split())
    mots_b = set(b.lower().split())
    if not mots_a or not mots_b:
        return 0.0
    return len(mots_a & mots_b) / len(mots_a | mots_b)

def selection_diverse(pensees_notes, k=2, seuil_similarite=0.6):
    """
    pensees_notes : liste de (pensee, note) deja triee par note decroissante.
    Renvoie k pensees en evitant les doublons semantiques.
    """
    pool = []
    # TODO etudiant : parcourir pensees_notes dans l'ordre ; n'ajouter une pensee
    # que si elle est suffisamment differente de toutes les deja gardees.
    return pool

ex_pensees = [
    ("Je suppose x=0 sans raison.", 9),
    ("Je suppose x=0 sans justifier.", 8),
    ("Posons L la largeur, l'aire vaut L*(100-2L).", 8),
]
print("Exercice 2 - pool divers sur donnees fictives :", selection_diverse(ex_pensees, k=2))

Exercice 2 - pool divers sur donnees fictives : []


### Exercice 3 — Verificateur execute dans la boucle Reflexion

Jusqu'ici le **Critique** est le LLM lui-meme. Rendez la boucle plus fiable en ajoutant un **verificateur execute** : demandez au modele d'ecrire le *code* de resolution, **executez-le**, et utilisez le resultat comme verite. Completez `reflexion_verifiee`.

In [16]:
# Indice :
#  1. Prompt = "ecris du code Python qui calcule la reponse, et termine par
#     `resultat = <la reponse>`."
#  2. Extraire le bloc Python (entre triple backticks ```python ... ```).
#  3. exec() dans un namespace isole, capturer les exceptions.
#  4. Si le code leve une exception -> critique = l'exception -> memoire -> retry.
#     Sinon -> retourner str(resultat).

def reflexion_verifiee(probleme, iterations=3):
    # TODO etudiant : implementer la boucle avec execution de code.
    resultat = None  # TODO etudiant
    return resultat

pred_exo3 = reflexion_verifiee(PROBLEMES[2])
print(f"Exercice 3 - reflexion verifiee sur '{PROBLEMES[2]['id']}' : {pred_exo3} "
      f"(attendu {PROBLEMES[2]['reponse']})")

Exercice 3 - reflexion verifiee sur 'machines' : None (attendu 5)


## 8. Conclusion et pont vers les agents

Vous avez construit, dans un seul notebook, les **quatre familles** du test-time scaling :

| Familie | Ce qu'on achete | Ce qu'on paie | Domaine naturel |
|---------|-----------------|---------------|-----------------|
| Best-of-N / Self-consistency | Robustesse, levée d'un biais de single-shot | N x tokens | Raisonnement multi-etapes |
| Tree-of-Thoughts | Resolution combinatoire | Explosion combinatoire | Recherche dans un espace d'etats |
| Self-Refine / Reflexion | Correction d'erreurs detectables | Boucles iteratives | Erreurs autocritiquables |
| Routeur adaptatif | Efficacite (bon effort au bon moment) | Un estimateur de difficulte | Tous, en production |

### Ou aller ensuite ?

- **Vers l'agentique** : le mode *Agentic* du depot ICR (4e mode, LangGraph) boucle ces techniques avec des **appels d'outils** — c'est l'objet du **notebook 4** (Function Calling) et du notebook 9 (Production Patterns).
- **Vers les modeles raisonnants** : comparez vos moteurs "faits maison" avec un modele raisonnant natif (notebook 8). Le test-time scaling interne fait essentiellement la meme chose, mais optimise dans les poids.
- **Vers la theorie** : Snell et al. (2024), *Scaling LLM Test-Time Compute Optimally*, formalisent *quand* il vaut mieux sampler plus vs chercher plus profond — exactement le routeur que nous avons code.

> **A retenir** : on n'a pas fini d'ameliorer un LLM quand on a bien prompte. On peut **investir du calcul au moment de l'inference** pour le faire raisonner mieux — a condition de choisir la technique adaptee a la structure du probleme.